In [ ]:
# Hybrid Movie Recommender System
# One-cell Jupyter Notebook version

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from scipy.sparse import csr_matrix, hstack
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ---------------------------
# 1. Setup
# ---------------------------
np.random.seed(42)
print("✅ Libraries loaded successfully")

# ---------------------------
# 2. Load Dataset
# ---------------------------
movies = pd.read_csv("movies.csv")

print("\n=== DATASET INFO ===")
print("Shape:", movies.shape)
print("Columns:", list(movies.columns))
display(movies.head())

# ---------------------------
# 3. Feature Engineering
# ---------------------------
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)", "", regex=True).str.strip()

movies["genres_text"] = (
    movies["genres"]
    .fillna("")
    .str.replace("|", " ", regex=False)
    .str.replace("(no genres listed)", "", regex=False)
    .str.strip()
)

ALL_GENRES = sorted(set(
    g for row in movies["genres"].dropna()
    for g in row.split("|")
    if g != "(no genres listed)"
))

print("\nUnique genres:", ALL_GENRES)

for g in ALL_GENRES:
    movies[f"g_{g}"] = movies["genres"].apply(lambda x: 1 if g in str(x).split("|") else 0)

print("✅ Feature engineering complete")
display(movies[["title", "clean_title", "year", "genres"]].head())

# ---------------------------
# 4. Build Item Profiles
# ---------------------------
tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z\-]+")
tfidf_matrix = tfidf.fit_transform(movies["genres_text"])

scaler = MinMaxScaler()
year_feat = scaler.fit_transform(movies[["year"]].fillna(movies["year"].median()))
year_sparse = csr_matrix(year_feat * 0.1)

ITEM_PROFILES = hstack([tfidf_matrix, year_sparse]).tocsr()

print("\n=== ITEM PROFILE INFO ===")
print("Item profile matrix shape:", ITEM_PROFILES.shape)
print("TF-IDF tokens:", list(tfidf.get_feature_names_out()))

# ---------------------------
# 5. Simulate Ratings
# ---------------------------
N_USERS = 500
N_MOVIES = len(movies)
DENSITY = 0.02
N_RATINGS = int(N_USERS * N_MOVIES * DENSITY)

genre_cols = [f"g_{g}" for g in ALL_GENRES]
genre_matrix = movies[genre_cols].values.astype(float)

user_genre_prefs = np.random.dirichlet(np.ones(len(ALL_GENRES)), size=N_USERS)
affinity = user_genre_prefs @ genre_matrix.T

flat_prob = affinity.flatten()
flat_prob = flat_prob / flat_prob.sum()

sampled = np.random.choice(len(flat_prob), size=N_RATINGS, replace=False, p=flat_prob)

uid = sampled // N_MOVIES
mid = sampled % N_MOVIES

base = 1.0 + 4.0 * affinity[uid, mid]
vals = np.clip(np.round(base + np.random.normal(0, 0.6, N_RATINGS), 1), 1.0, 5.0)

ratings_df = pd.DataFrame({
    "userId": uid,
    "movieIdx": mid,
    "movieId": movies["movieId"].values[mid],
    "rating": vals
})

print("\n=== SIMULATED RATINGS INFO ===")
print("Simulated ratings:", len(ratings_df))
print("Users:", ratings_df["userId"].nunique())
print("Movies rated:", ratings_df["movieId"].nunique())
print("Sparsity:", f"{1 - len(ratings_df)/(N_USERS*N_MOVIES):.3%}")
display(ratings_df.head())

# ---------------------------
# 6. EDA
# ---------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("MovieLens Dataset – Exploratory Analysis", fontsize=15, fontweight="bold")

gcounts = {g: movies[f"g_{g}"].sum() for g in ALL_GENRES}
gdf = pd.Series(gcounts).sort_values()

axes[0, 0].barh(gdf.index, gdf.values, color=cm.tab20(np.linspace(0, 1, len(gdf))))
axes[0, 0].set_title("(A) Movies per Genre", fontweight="bold")
axes[0, 0].set_xlabel("Count")
axes[0, 0].grid(axis="x", alpha=0.3)

dec = movies.dropna(subset=["year"]).copy()
dec["decade"] = (dec["year"] // 10 * 10).astype(int)
dc = dec.groupby("decade").size()

axes[0, 1].bar(dc.index.astype(str), dc.values, color=cm.viridis(np.linspace(0.2, 0.9, len(dc))), width=0.7)
axes[0, 1].set_title("(B) Movies per Decade", fontweight="bold")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].grid(axis="y", alpha=0.3)

axes[1, 0].hist(ratings_df["rating"], bins=30, color="steelblue", edgecolor="white")
axes[1, 0].axvline(ratings_df["rating"].mean(), color="crimson", lw=2, label=f"Mean = {ratings_df['rating'].mean():.2f}")
axes[1, 0].set_title("(C) Rating Distribution", fontweight="bold")
axes[1, 0].set_xlabel("Rating")
axes[1, 0].legend()
axes[1, 0].grid(axis="y", alpha=0.3)

rpu = ratings_df.groupby("userId").size().sort_values(ascending=False).values
axes[1, 1].plot(rpu, color="darkorange", lw=1.5)
axes[1, 1].fill_between(range(len(rpu)), rpu, alpha=0.2, color="darkorange")
axes[1, 1].set_title("(D) Long-Tail: Ratings per User", fontweight="bold")
axes[1, 1].set_xlabel("User rank")
axes[1, 1].set_ylabel("# Ratings")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ---------------------------
# 7. Content-Based Filtering
# ---------------------------
print("\nComputing item-item cosine similarity...")
ITEM_SIM = cosine_similarity(ITEM_PROFILES, ITEM_PROFILES)
title_to_idx = pd.Series(movies.index, index=movies["clean_title"]).drop_duplicates()
print("Item similarity matrix shape:", ITEM_SIM.shape)
print("✅ CBF ready")

def build_user_profile_cbf(user_id, top_n_rated=30):
    user_ratings = ratings_df[ratings_df["userId"] == user_id].nlargest(top_n_rated, "rating")
    if user_ratings.empty:
        return None

    idxs = user_ratings["movieIdx"].values
    weights = user_ratings["rating"].values.astype(float)
    weights = weights / weights.sum()

    profile = ITEM_PROFILES[idxs].multiply(weights[:, None]).sum(axis=0)
    return np.asarray(profile).flatten()

def recommend_cbf(user_id, n=10):
    user_vec = build_user_profile_cbf(user_id)
    if user_vec is None:
        return pd.DataFrame({"error": ["User has no ratings"]})

    seen_idxs = set(ratings_df[ratings_df["userId"] == user_id]["movieIdx"].values)
    sims = cosine_similarity(user_vec.reshape(1, -1), ITEM_PROFILES)[0]

    ranked = sorted(
        [(i, s) for i, s in enumerate(sims) if i not in seen_idxs],
        key=lambda x: x[1],
        reverse=True
    )[:n]

    return pd.DataFrame([
        {
            "title": movies.loc[i, "clean_title"],
            "genres": movies.loc[i, "genres"],
            "year": movies.loc[i, "year"],
            "cbf_score": round(float(s), 4),
            "movieIdx": i
        }
        for i, s in ranked
    ])

print("\n=== CBF Recommendations for User 0 ===")
cbf_demo = recommend_cbf(0, n=10)
display(cbf_demo)

# ---------------------------
# 8. Collaborative Filtering
# ---------------------------
R = np.zeros((N_USERS, N_MOVIES), dtype=np.float32)
for _, row in ratings_df.iterrows():
    R[int(row["userId"]), int(row["movieIdx"])] = row["rating"]

print("\n=== UTILITY MATRIX INFO ===")
print("Utility matrix shape:", R.shape)
print("Non-zero entries:", int((R > 0).sum()))
print("Sparsity:", f"{1 - (R > 0).mean():.3%}")

user_means = np.true_divide(R.sum(axis=1), (R > 0).sum(axis=1).clip(1))
R_centered = R.copy()

for u in range(N_USERS):
    mask = R[u] > 0
    R_centered[u, mask] -= user_means[u]

K = min(50, min(R_centered.shape) - 1)
print(f"\nRunning SVD with k = {K} latent factors...")

U_svd, sigma, Vt_svd = svds(csr_matrix(R_centered), k=K)
sigma = np.diag(sigma)

R_pred_svd = np.clip(U_svd @ sigma @ Vt_svd + user_means[:, np.newaxis], 1.0, 5.0)

known_mask = R > 0
rmse_svd = np.sqrt(mean_squared_error(R[known_mask], R_pred_svd[known_mask]))

print(f"RMSE on known ratings: {rmse_svd:.4f}")
print("✅ SVD model trained")

print("\nComputing user-user similarity...")
USER_SIM = cosine_similarity(csr_matrix(R))
np.fill_diagonal(USER_SIM, 0)
print("User similarity matrix shape:", USER_SIM.shape)

def recommend_user_based_cf(user_id, n=10, k_neighbors=30):
    sim_scores = USER_SIM[user_id]
    top_k_users = np.argsort(sim_scores)[::-1][:k_neighbors]
    top_k_sims = sim_scores[top_k_users]

    seen_idxs = set(np.where(R[user_id] > 0)[0])
    pred_scores = {}

    for movie_idx in range(N_MOVIES):
        if movie_idx in seen_idxs:
            continue

        neighbor_ratings = R[top_k_users, movie_idx]
        mask = neighbor_ratings > 0

        if mask.sum() == 0:
            continue

        w = top_k_sims[mask]
        if np.abs(w).sum() == 0:
            continue

        pred_scores[movie_idx] = np.dot(neighbor_ratings[mask], w) / np.abs(w).sum()

    ranked = sorted(pred_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    return pd.DataFrame([
        {
            "title": movies.loc[i, "clean_title"],
            "genres": movies.loc[i, "genres"],
            "year": movies.loc[i, "year"],
            "ubcf_score": round(float(s), 4),
            "movieIdx": i
        }
        for i, s in ranked
    ])

def recommend_svd_cf(user_id, n=10):
    seen_idxs = set(np.where(R[user_id] > 0)[0])

    preds = sorted(
        [(i, R_pred_svd[user_id, i]) for i in range(N_MOVIES) if i not in seen_idxs],
        key=lambda x: x[1],
        reverse=True
    )[:n]

    return pd.DataFrame([
        {
            "title": movies.loc[i, "clean_title"],
            "genres": movies.loc[i, "genres"],
            "year": movies.loc[i, "year"],
            "cf_score": round(float(s), 4),
            "movieIdx": i
        }
        for i, s in preds
    ])

print("\n=== User-Based CF Recommendations for User 0 ===")
display(recommend_user_based_cf(0, n=10))

# ---------------------------
# 9. Hybrid Recommender
# ---------------------------
def recommend_hybrid(user_id, n=10, alpha=0.5):
    seen_idxs = set(np.where(R[user_id] > 0)[0])
    unseen = [i for i in range(N_MOVIES) if i not in seen_idxs]

    user_vec = build_user_profile_cbf(user_id)
    if user_vec is None:
        alpha = 0.0
        cbf_scores_all = np.zeros(N_MOVIES)
    else:
        cbf_scores_all = cosine_similarity(user_vec.reshape(1, -1), ITEM_PROFILES)[0]

    cf_raw = R_pred_svd[user_id]
    cf_min, cf_max = cf_raw.min(), cf_raw.max()
    cf_scores_all = (cf_raw - cf_min) / (cf_max - cf_min + 1e-9)

    scores = {
        i: alpha * cbf_scores_all[i] + (1 - alpha) * cf_scores_all[i]
        for i in unseen
    }

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]

    return pd.DataFrame([
        {
            "title": movies.loc[i, "clean_title"],
            "genres": movies.loc[i, "genres"],
            "year": int(movies.loc[i, "year"]) if pd.notna(movies.loc[i, "year"]) else "N/A",
            "cbf_score": round(float(cbf_scores_all[i]), 4),
            "cf_score": round(float(cf_scores_all[i]), 4),
            "hybrid_score": round(float(s), 4),
            "movieIdx": i
        }
        for i, s in ranked
    ])

print("\n=== Hybrid Recommendations for User 0 (alpha = 0.5) ===")
hybrid_demo = recommend_hybrid(0, n=10, alpha=0.5)
display(hybrid_demo)

# ---------------------------
# 10. User Profile Inspection
# ---------------------------
def inspect_user(user_id):
    user_ratings = ratings_df[ratings_df["userId"] == user_id].merge(
        movies[["movieId", "clean_title", "genres", "year"]],
        on="movieId"
    ).sort_values("rating", ascending=False)

    print(f"\n--- User {user_id} Rating History ---")
    display(user_ratings[["clean_title", "genres", "rating"]].head(10))

    genre_profile = {g: 0.0 for g in ALL_GENRES}

    for _, r in user_ratings.iterrows():
        for g in str(r["genres"]).split("|"):
            if g in genre_profile:
                genre_profile[g] += r["rating"]

    total = sum(genre_profile.values())
    genre_profile = {k: v / total for k, v in genre_profile.items() if v > 0}

    gdf = pd.Series(genre_profile).sort_values()

    plt.figure(figsize=(9, 4))
    plt.barh(gdf.index, gdf.values, color=cm.RdYlGn(np.linspace(0.2, 0.9, len(gdf))))
    plt.title(f"User {user_id} – Genre Profile (rating-weighted)", fontweight="bold")
    plt.xlabel("Normalized preference score")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

inspect_user(0)

# ---------------------------
# 11. Compare Methods
# ---------------------------
USER_ID = 0
TOP_N = 8

cbf_recs = recommend_cbf(USER_ID, n=TOP_N)
cf_recs = recommend_svd_cf(USER_ID, n=TOP_N)
hybrid_recs = recommend_hybrid(USER_ID, n=TOP_N, alpha=0.5)

print("\n" + "=" * 90)
print(f"USER {USER_ID} — Top {TOP_N} Recommendations")
print("=" * 90)

print("\nCONTENT-BASED FILTERING:")
display(cbf_recs[["title", "genres", "cbf_score"]])

print("\nCOLLABORATIVE FILTERING (SVD):")
display(cf_recs[["title", "genres", "cf_score"]])

print("\nHYBRID (alpha = 0.5):")
display(hybrid_recs[["title", "genres", "cbf_score", "cf_score", "hybrid_score"]])

# ---------------------------
# 12. Tune Alpha
# ---------------------------
print("\n=== Alpha Tuning ===")
for alpha, label in zip(
    [0.0, 0.25, 0.5, 0.75, 1.0],
    ["Pure CF", "CF-heavy", "Balanced", "CBF-heavy", "Pure CBF"]
):
    recs = recommend_hybrid(USER_ID, n=5, alpha=alpha)
    titles = " | ".join(recs["title"].values[:5])
    print(f"alpha = {alpha:.2f} ({label}): {titles}")

# ---------------------------
# 13. Evaluation: RMSE & MAE
# ---------------------------
train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

R_train = np.zeros((N_USERS, N_MOVIES), dtype=np.float32)
for _, row in train_df.iterrows():
    R_train[int(row["userId"]), int(row["movieIdx"])] = row["rating"]

um_train = np.true_divide(R_train.sum(axis=1), (R_train > 0).sum(axis=1).clip(1))
R_tr_cen = R_train.copy()

for u in range(N_USERS):
    mask = R_train[u] > 0
    R_tr_cen[u, mask] -= um_train[u]

K_eval = min(50, min(R_tr_cen.shape) - 1)
U_t, s_t, Vt_t = svds(csr_matrix(R_tr_cen), k=K_eval)
R_pred_test = np.clip(U_t @ np.diag(s_t) @ Vt_t + um_train[:, None], 1.0, 5.0)

true_r = test_df["rating"].values
pred_r = R_pred_test[
    test_df["userId"].values.astype(int),
    test_df["movieIdx"].values.astype(int)
]

print("\n=== TEST EVALUATION ===")
print(f"RMSE: {np.sqrt(mean_squared_error(true_r, pred_r)):.4f}")
print(f"MAE : {np.mean(np.abs(true_r - pred_r)):.4f}")

# ---------------------------
# 14. Precision@K & Recall@K
# ---------------------------
def precision_recall_at_k(user_id, k=10, threshold=3.5):
    relevant = set(
        test_df[
            (test_df["userId"] == user_id) &
            (test_df["rating"] >= threshold)
        ]["movieIdx"].values
    )

    if len(relevant) == 0:
        return None, None

    recs = recommend_hybrid(user_id, n=k, alpha=0.5)["movieIdx"].values
    n_hit = len(set(recs) & relevant)

    return n_hit / k, n_hit / len(relevant)

results = [precision_recall_at_k(u) for u in range(100)]
results = [(p, r) for p, r in results if p is not None]

print("\n=== RANKING EVALUATION ===")
print(f"Mean Precision@10: {np.mean([p for p, _ in results]):.4f}")
print(f"Mean Recall@10   : {np.mean([r for _, r in results]):.4f}")

# ---------------------------
# 15. Precision/Recall vs Alpha
# ---------------------------
alpha_vals = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
eval_rows = []

for alpha in alpha_vals:
    precs, recs_l = [], []

    for u in range(100):
        relevant = set(
            test_df[
                (test_df["userId"] == u) &
                (test_df["rating"] >= 3.5)
            ]["movieIdx"].values
        )

        if len(relevant) == 0:
            continue

        recs_u = recommend_hybrid(u, n=10, alpha=alpha)["movieIdx"].values
        hit = len(set(recs_u) & relevant)

        precs.append(hit / 10)
        recs_l.append(hit / len(relevant))

    eval_rows.append({
        "alpha": alpha,
        "precision": np.mean(precs) if len(precs) else 0,
        "recall": np.mean(recs_l) if len(recs_l) else 0
    })

eval_df = pd.DataFrame(eval_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Hybrid Recommender Evaluation – Effect of Alpha", fontsize=13, fontweight="bold")

axes[0].plot(eval_df["alpha"], eval_df["precision"], "o-", color="steelblue", lw=2)
axes[0].set_title("Precision@10 vs Alpha")
axes[0].set_xlabel("Alpha")
axes[0].set_ylabel("Precision@10")
axes[0].grid(alpha=0.3)

axes[1].plot(eval_df["alpha"], eval_df["recall"], "s-", color="darkorange", lw=2)
axes[1].set_title("Recall@10 vs Alpha")
axes[1].set_xlabel("Alpha")
axes[1].set_ylabel("Recall@10")
axes[1].grid(alpha=0.3)

axes[2].plot(eval_df["recall"], eval_df["precision"], "D-", color="seagreen", lw=2)
for _, row in eval_df.iterrows():
    axes[2].annotate(
        f"a={row['alpha']}",
        (row["recall"], row["precision"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=7
    )
axes[2].set_title("Precision–Recall Curve")
axes[2].set_xlabel("Recall@10")
axes[2].set_ylabel("Precision@10")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(eval_df)

# ---------------------------
# 16. Cold-Start Demo
# ---------------------------
def cold_start_demo(favorite_genres, n=8):
    tokens = " ".join(favorite_genres)
    pseudo = tfidf.transform([tokens])
    pseudo_full = hstack([pseudo, csr_matrix([[0.5]])]).toarray().flatten()

    w = ITEM_PROFILES.shape[1]
    if len(pseudo_full) < w:
        pseudo_full = np.pad(pseudo_full, (0, w - len(pseudo_full)))
    else:
        pseudo_full = pseudo_full[:w]

    sims = cosine_similarity(pseudo_full.reshape(1, -1), ITEM_PROFILES)[0]
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)[:n]

    print(f"\nNew user who likes: {favorite_genres}")
    for i, s in ranked:
        print(f"{movies.loc[i, 'clean_title']:<45} score = {s:.4f}")

cold_start_demo(["Action", "Sci-Fi", "Thriller"])
cold_start_demo(["Animation", "Comedy", "Children"])
cold_start_demo(["Drama", "Romance"])

# ---------------------------
# 17. Summary
# ---------------------------
print("\n=== SUMMARY ===")
print("CBF strengths   : explainable, handles item cold-start")
print("CBF weaknesses  : over-specialization, weak for new users")
print("CF strengths    : captures latent patterns")
print("CF weaknesses   : cold-start, popularity bias")
print("Hybrid strengths: combines both methods")
print("Hybrid weakness : needs alpha tuning")
print("\n✅ Notebook finished successfully")